# Dev-Only Aspect Extraction Length Analysis

## 1. Imports

In [1]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="paper")
except Exception:
    sns = None

plt.rcParams.update({
    "figure.dpi": 140,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
})

## 2. Paths

In [ ]:
DATA_DIR = Path(os.environ.get("DATA_DIR", "/kaggle/input/datasets/anisoaraana/aspect-aware-narrative-similarity"))
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEV_TRACK_A_PATH = Path(os.environ.get("DEV_TRACK_A_PATH", str(DATA_DIR / "dev_track_a.jsonl")))

ASPECT_CANDIDATES = {
    "V1 verbose separate prompts": [
        os.environ.get("ASPECTS_V1_PATH", "/kaggle/input/datasets/anisoaraana/aspect-aware-narrative-similarity/aspects_cache_v1.json" ),
        DATA_DIR / "aspects_cache_v1.json",
    ],
    "V2 structured numbered CoA": [
        os.environ.get("ASPECTS_V2_PATH", "/kaggle/input/datasets/anisoaraana/aspect-aware-narrative-similarity/aspects_cache_v2.json"),
        DATA_DIR / "aspects_cache_v2.json",
    ],
    "V3 compact constrained": [
        os.environ.get("ASPECTS_V3_PATH", "/kaggle/input/datasets/anisoaraana/aspect-aware-narrative-similarity/aspects_cache_v3.json"),
        DATA_DIR / "aspects_cache_v3.json",
    ],
}

def resolve_first_existing(candidates):
    for candidate in candidates:
        if not candidate:
            continue
        path = Path(candidate)
        if path.exists():
            return path
    return None

ASPECT_PATHS = {name: resolve_first_existing(paths) for name, paths in ASPECT_CANDIDATES.items()}

path_status = pd.DataFrame([
    {"resource": "dev_track_a", "path": str(DEV_TRACK_A_PATH), "exists": DEV_TRACK_A_PATH.exists()},
    *[{"resource": name, "path": str(path) if path else "NOT FOUND", "exists": bool(path and path.exists())}
      for name, path in ASPECT_PATHS.items()]
])
path_status

## 3. Utility Functions

In [ ]:
ASPECTS = ["coa", "outcomes", "theme"]

ASPECT_SYNONYMS = {
    "coa": ["coa", "course_of_action"],
    "outcomes": ["outcomes", "outcome"],
    "theme": ["theme", "abstract_theme"],
}

def norm_text(text):
    return " ".join(str(text).split())

def word_count(text):
    text = str(text or "").strip()
    if not text:
        return 0
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))

def read_json_or_jsonl(path):
    path = Path(path)
    raw = path.read_text(encoding="utf-8").strip()
    if not raw:
        return []
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

def load_dev_texts(dev_path):
    rows = read_json_or_jsonl(dev_path)
    texts = []
    for row in rows:
        for col in ["anchor_text", "text_a", "text_b"]:
            value = row.get(col)
            if value:
                texts.append(str(value))
    by_key = {}
    for text in texts:
        by_key.setdefault(norm_text(text), text)
    return by_key, rows

def iter_aspect_records(raw):
    """Yield (story_text, payload) from common aspect-cache formats."""
    if isinstance(raw, dict):
        if isinstance(raw.get("records"), list):
            for item in raw["records"]:
                if isinstance(item, dict):
                    payload = item.get("aspects") if isinstance(item.get("aspects"), dict) else item
                    text = item.get("text") or item.get("raw_text") or item.get("story") or item.get("story_text")
                    if text and isinstance(payload, dict):
                        yield text, payload
        else:
            for key, value in raw.items():
                if isinstance(value, dict):
                    payload = value.get("aspects") if isinstance(value.get("aspects"), dict) else value
                    yield key, payload
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            payload = item.get("aspects") if isinstance(item.get("aspects"), dict) else item
            text = item.get("text") or item.get("raw_text") or item.get("story") or item.get("story_text")
            if text and isinstance(payload, dict):
                yield text, payload

def get_aspect(payload, aspect):
    if not isinstance(payload, dict):
        return ""
    for key in ASPECT_SYNONYMS[aspect]:
        if key in payload and payload[key] is not None:
            return str(payload[key]).strip()
    return ""

def load_aspect_cache(path):
    raw = read_json_or_jsonl(path)
    cache = {}
    for text, payload in iter_aspect_records(raw):
        cache[norm_text(text)] = payload
    return cache

## 4. Build Dev-Only Aspect Length Table

In [ ]:
dev_texts, dev_rows = load_dev_texts(DEV_TRACK_A_PATH)
print(f"Dev triples: {len(dev_rows)}")
print(f"Unique dev stories: {len(dev_texts)}")

missing_versions = [name for name, path in ASPECT_PATHS.items() if path is None]
if missing_versions:
    print("Missing aspect files:")
    for name in missing_versions:
        print("  -", name)
    print("Set ASPECTS_V1_PATH / ASPECTS_V2_PATH / ASPECTS_V3_PATH or edit ASPECT_CANDIDATES above.")

In [ ]:
records = []
coverage_records = []

for version, path in ASPECT_PATHS.items():
    if path is None:
        continue
    cache = load_aspect_cache(path)
    matched_story_count = 0
    for story_key, original_text in dev_texts.items():
        payload = cache.get(story_key)
        if payload is not None:
            matched_story_count += 1
        for aspect in ASPECTS:
            aspect_text = get_aspect(payload, aspect) if payload is not None else ""
            records.append({
                "version": version,
                "aspect": aspect,
                "story_key": story_key,
                "present": bool(aspect_text.strip()),
                "chars": len(aspect_text),
                "words": word_count(aspect_text),
                "aspect_text": aspect_text,
            })
    for aspect in ASPECTS:
        aspect_hits = sum(
            bool(get_aspect(cache.get(story_key), aspect).strip())
            for story_key in dev_texts
        )
        coverage_records.append({
            "version": version,
            "aspect": aspect,
            "story_hits": aspect_hits,
            "total_dev_stories": len(dev_texts),
            "hit_rate": aspect_hits / max(len(dev_texts), 1),
            "cache_path": str(path),
        })
    coverage_records.append({
        "version": version,
        "aspect": "any_payload",
        "story_hits": matched_story_count,
        "total_dev_stories": len(dev_texts),
        "hit_rate": matched_story_count / max(len(dev_texts), 1),
        "cache_path": str(path),
    })

length_df = pd.DataFrame(records)
coverage_df = pd.DataFrame(coverage_records)

length_csv = OUT_DIR / "dev_aspect_extraction_lengths_long.csv"
coverage_csv = OUT_DIR / "dev_aspect_extraction_coverage.csv"
length_df.to_csv(length_csv, index=False)
coverage_df.to_csv(coverage_csv, index=False)

print(f"Saved long length table: {length_csv}")
print(f"Saved coverage table: {coverage_csv}")
coverage_df

## 5. Summary Statistics

In [ ]:
present_df = length_df[length_df["present"]].copy()

summary_df = (
    present_df
    .groupby(["version", "aspect"], as_index=False)
    .agg(
        n=("story_key", "count"),
        mean_chars=("chars", "mean"),
        median_chars=("chars", "median"),
        std_chars=("chars", "std"),
        mean_words=("words", "mean"),
        median_words=("words", "median"),
        std_words=("words", "std"),
    )
)

summary_path = OUT_DIR / "dev_aspect_extraction_length_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Saved summary table: {summary_path}")
summary_df

## 6. Plot Absolute Lengths by Version and Aspect

These plots show how verbose each extraction version is for each aspect type.

In [ ]:
VERSION_ORDER = [name for name in ASPECT_PATHS.keys() if name in set(present_df["version"])]
ASPECT_ORDER = ["coa", "outcomes", "theme"]

def plot_absolute_lengths(metric="words"):
    ylabel = "Word count" if metric == "words" else "Character count"
    title = f"Dev-set extracted aspect length by extraction version ({ylabel.lower()})"
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)

    if sns is not None:
        sns.boxplot(
            data=present_df,
            x="aspect",
            y=metric,
            hue="version",
            order=ASPECT_ORDER,
            hue_order=VERSION_ORDER,
            showfliers=False,
            ax=axes[0],
        )
        axes[0].set_title("Distribution, outliers hidden")
        axes[0].set_xlabel("Aspect")
        axes[0].set_ylabel(ylabel)

        try:
            sns.barplot(
                data=present_df,
                x="aspect",
                y=metric,
                hue="version",
                order=ASPECT_ORDER,
                hue_order=VERSION_ORDER,
                errorbar="sd",
                ax=axes[1],
            )
        except TypeError:
            sns.barplot(
                data=present_df,
                x="aspect",
                y=metric,
                hue="version",
                order=ASPECT_ORDER,
                hue_order=VERSION_ORDER,
                ci="sd",
                ax=axes[1],
            )
        axes[1].set_title("Mean length with standard deviation")
        axes[1].set_xlabel("Aspect")
        axes[1].set_ylabel(ylabel)
    else:
        pivot = summary_df.pivot(index="aspect", columns="version", values=f"mean_{metric}").reindex(ASPECT_ORDER)
        pivot.plot(kind="bar", ax=axes[0])
        axes[0].set_title("Mean length")
        axes[0].set_ylabel(ylabel)
        axes[1].axis("off")

    for ax in axes:
        ax.grid(axis="y", alpha=0.25)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(title="Extraction version", loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0)

    fig.suptitle(title, y=1.03, fontsize=12)
    out_path = OUT_DIR / f"dev_aspect_absolute_{metric}.png"
    fig.savefig(out_path, bbox_inches="tight")
    print(f"Saved plot: {out_path}")
    return fig

plot_absolute_lengths("words")
plot_absolute_lengths("chars")

## 7. Plot Differences Relative to Version 1

How much longer or shorter are versions 2 and 3 compared with version 1, for the same dev stories and same aspect type?

In [ ]:
def make_delta_df(metric="words", reference_version=None):
    if reference_version is None:
        reference_version = VERSION_ORDER[0] if VERSION_ORDER else None
    if reference_version is None:
        return pd.DataFrame()

    wide = (
        length_df
        .pivot_table(index=["story_key", "aspect"], columns="version", values=metric, aggfunc="first")
        .reset_index()
    )
    if reference_version not in wide.columns:
        return pd.DataFrame()

    rows = []
    for version in VERSION_ORDER:
        if version == reference_version or version not in wide.columns:
            continue
        tmp = wide[["story_key", "aspect", reference_version, version]].dropna().copy()
        tmp["reference_version"] = reference_version
        tmp["comparison_version"] = version
        tmp["delta"] = tmp[version] - tmp[reference_version]
        tmp["metric"] = metric
        rows.append(tmp[["story_key", "aspect", "reference_version", "comparison_version", "metric", "delta"]])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

delta_words = make_delta_df("words")
delta_chars = make_delta_df("chars")
delta_df = pd.concat([delta_words, delta_chars], ignore_index=True)

delta_path = OUT_DIR / "dev_aspect_extraction_length_deltas.csv"
delta_df.to_csv(delta_path, index=False)
print(f"Saved delta table: {delta_path}")
delta_df.head()

In [ ]:
def plot_deltas(metric="words"):
    data = delta_df[delta_df["metric"] == metric].copy()
    if data.empty:
        print("No delta data available. Check that at least two aspect versions were loaded.")
        return None
    ylabel = "Difference in words" if metric == "words" else "Difference in characters"
    ref = data["reference_version"].iloc[0]

    fig, ax = plt.subplots(figsize=(8.2, 4.2), constrained_layout=True)
    if sns is not None:
        try:
            sns.barplot(
                data=data,
                x="aspect",
                y="delta",
                hue="comparison_version",
                order=ASPECT_ORDER,
                errorbar="sd",
                ax=ax,
            )
        except TypeError:
            sns.barplot(
                data=data,
                x="aspect",
                y="delta",
                hue="comparison_version",
                order=ASPECT_ORDER,
                ci="sd",
                ax=ax,
            )
    else:
        pivot = data.groupby(["aspect", "comparison_version"])["delta"].mean().unstack()
        pivot.reindex(ASPECT_ORDER).plot(kind="bar", ax=ax)

    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_title(f"Mean length difference relative to {ref}")
    ax.set_xlabel("Aspect")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Compared version", loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0)

    out_path = OUT_DIR / f"dev_aspect_delta_vs_v1_{metric}.png"
    fig.savefig(out_path, bbox_inches="tight")
    print(f"Saved plot: {out_path}")
    return fig

plot_deltas("words")
plot_deltas("chars")

## 8. Results Table

In [ ]:
thesis_table = summary_df.copy()
for col in ["mean_chars", "median_chars", "std_chars", "mean_words", "median_words", "std_words"]:
    thesis_table[col] = thesis_table[col].round(2)

thesis_table_path = OUT_DIR / "dev_aspect_extraction_length_thesis_table.csv"
thesis_table.to_csv(thesis_table_path, index=False)
print(f"Saved thesis table: {thesis_table_path}")
thesis_table